In [7]:
import os
import glob
import rasterio
import geopandas as gpd
'''
# 1. Load the shapefile directly (GeoPandas automatically reads the UTM geometry and CRS)
# points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/gce_sites_zone_pixels_centroids/gce_sites_zone_pixels_centroids.shp")
# points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/gce_sites_zone_pixels_centroids2/gce_sites_zone_pixels_centroids2.shp")

# 2. Get the native CRS of your biomass models
raster_files = sorted(glob.glob('F:/Biomass tiffs/Biomass modeling/Georgia/Georgia model only/LOGO_final/*.tif'))
with rasterio.open(raster_files[0]) as src:
    raster_crs = src.crs

# 3. Project the points to match the raster if they don't already
if points_gdf.crs != raster_crs:
    points_gdf = points_gdf.to_crs(raster_crs)

# Extract coordinates as a list of (x, y) tuples
coord_tuples = [(pt.x, pt.y) for pt in points_gdf.geometry]

# 4. Loop through the years and sample the data
for raster_path in raster_files:
    year = os.path.basename(raster_path).split("_")[-1].replace(".tif", "")
    print(f"Extracting data for year: {year}...")
    
    with rasterio.open(raster_path) as src:
        # Sample the raster and append directly to the GeoDataFrame
        points_gdf[f"biomass_{year}"] = [val[0] for val in src.sample(coord_tuples)]

# 5. Save out the clean results
# You can save it back out as a flat CSV for them, or even a new shapefile/GeoJSON
points_gdf.drop(columns='geometry').to_csv("biomass_extractions_final.csv", index=False)
print("Done!")'''''

'\n# 1. Load the shapefile directly (GeoPandas automatically reads the UTM geometry and CRS)\n# points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/gce_sites_zone_pixels_centroids/gce_sites_zone_pixels_centroids.shp")\n# points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/gce_sites_zone_pixels_centroids2/gce_sites_zone_pixels_centroids2.shp")\n\n# 2. Get the native CRS of your biomass models\nraster_files = sorted(glob.glob(\'F:/Biomass tiffs/Biomass modeling/Georgia/Georgia model only/LOGO_final/*.tif\'))\nwith rasterio.open(raster_files[0]) as src:\n    raster_crs = src.crs\n\n# 3. Project the points to match the raster if they don\'t already\nif points_gdf.crs != raster_crs:\n    points_gdf = points_gdf.to_crs(raster_crs)\n\n# Extract coordinates as a list of (x, y) tuples\ncoord_tuples = [(pt.x, pt.y) for pt in points_gdf.geometry]\n\n# 4. Loop through the years and sample the data\nfor raster_path in raster_fi

In [2]:
points_gdf

NameError: name 'points_gdf' is not defined

In [ ]:
##Exporting extraction

out_dir = os.path.expanduser('~/Downloads')
out_csv = os.path.join(out_dir, 'rest_trends.csv')

#points_gdf.to_csv(out_csv)

In [2]:
##Window extraction around restoration points
import rasterio
from rasterio.windows import Window
import numpy as np
import pandas as pd
import geopandas as gpd
import glob
import os

# 1. Setup
points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/Restoration sites/Rest_sites.shp")
raster_files = sorted(glob.glob('F:/Biomass tiffs/Biomass modeling/Georgia/Georgia model only/LOGO_final/*.tif'))

with rasterio.open(raster_files[0]) as src:
    raster_crs = src.crs
    if points_gdf.crs != raster_crs:
        points_gdf = points_gdf.to_crs(raster_crs)

# Define window size (e.g., 2x2)
win_size = 4
pad = win_size // 4

# 2. Loop through files
for raster_path in raster_files:
    year = os.path.basename(raster_path).split("_")[-1].replace(".tif", "")
    print(f"Extracting 4x4 mean for year: {year}...")
    
    with rasterio.open(raster_path) as src:
        results = []
        for idx, row in points_gdf.iterrows():
            # Get pixel coordinates
            row_idx, col_idx = src.index(row.geometry.x, row.geometry.y)
            
            # Define the window
            window = Window(col_idx - pad, row_idx - pad, win_size, win_size)
            
            # Read the window data
            data = src.read(1, window=window)
            
            # Mask out nodata/NaN values
            # Replace 'src.nodata' with np.nan if your tiff uses actual NaNs
            if src.nodata is not None:
                data = np.where(data == src.nodata, np.nan, data)
                
            # Calculate mean ignoring NaNs
            if np.all(np.isnan(data)):
                results.append(np.nan)
            else:
                results.append(np.nanmean(data))
        
        points_gdf[f"biomass_{year}"] = results

# 3. Save
points_gdf.drop(columns='geometry').to_csv("biomass_extractions_4x4.csv", index=False)
print("Done!")

ModuleNotFoundError: No module named 'rasterio.windows'

In [5]:
points_gdf

,Site,Latitude,Longitude,Type,geometry,biomass_1984,biomass_1985,biomass_1986,biomass_1987,biomass_1988,...,biomass_2014,biomass_2015,biomass_2016,biomass_2017,biomass_2018,biomass_2019,biomass_2020,biomass_2021,biomass_2022,biomass_2023
0,Little Cumberland Island,30.950135,-81.421247,LS,POINT (-81.42125 30.95014),845.632202,933.813477,799.116638,824.808472,816.119751,...,840.648071,983.949829,818.740051,980.403198,818.742065,829.755676,835.821411,940.861145,815.248779,786.960815
1,"Fortson Pond, Jekyll Island",31.053372,-81.414579,LS,POINT (-81.41458 31.05337),885.162720,988.716492,881.033081,889.252563,959.028687,...,906.148071,1055.030029,931.354492,1023.133728,882.690186,872.310669,1063.300171,1039.480347,970.523315,873.720276
2,"DNR’s Coastal Regional Headquarters, Brunswick",31.122932,-81.478103,LS,POINT (-81.4781 31.12293),368.049713,535.406372,373.745758,409.829102,460.569641,...,365.030334,562.932129,371.112610,528.135437,369.428650,365.227844,545.489319,531.824463,388.307556,386.003021
3,"Beach Road Bridge, Little St. Simons Island",31.249670,-81.293518,LS,POINT (-81.29352 31.24967),569.847717,835.799194,575.408325,602.188904,625.453979,...,764.715088,977.683594,604.434082,692.834229,715.331909,734.273315,979.992920,988.187378,739.651367,751.543274
4,"Cannon’s Point Preserve, St. Simon’s Island",31.260038,-81.339778,LS,POINT (-81.33978 31.26004),622.550415,706.866638,590.903015,622.860168,731.036987,...,608.711792,729.271484,605.489380,596.202515,651.209106,661.011841,826.275513,781.883850,625.793518,667.444153
5,"Mosquito Creek, Little St. Simon’s Island",31.260078,-81.301105,LS,POINT (-81.3011 31.26008),634.806396,866.574036,627.699280,649.412292,730.943176,...,714.870056,943.925476,724.032410,703.487488,738.799805,731.459656,969.149475,918.246338,705.757324,715.641357
6,"Long Tabby, Sapelo Island",31.431955,-81.282659,LS,POINT (-81.28266 31.43196),869.103271,784.144775,680.016968,1002.779724,842.643127,...,761.466064,966.029480,570.443848,1159.304199,661.316833,669.801758,927.405029,565.239319,1333.237671,1198.355103
7,"Ashantilly, Sapelo Island",31.435565,-81.280004,LS,POINT (-81.28 31.43557),1134.372681,1157.219482,1138.954712,1134.064209,1057.723755,...,1082.276001,1230.273682,1008.475281,955.304810,1136.453125,1080.640747,1095.369873,1180.381348,1002.009949,1136.510498
8,"Skidaway Island State Park, Skidaway Island",31.951577,-81.055498,LS,POINT (-81.0555 31.95158),756.271606,781.118286,919.217529,723.581116,929.094116,...,643.707275,907.423828,691.308533,597.531860,612.939209,529.747559,690.808472,645.660156,674.531311,671.139526
9,"Burton 4-H, Tybee Island",32.007732,-80.851325,LS,POINT (-80.85132 32.00773),622.936157,639.629028,869.150696,625.258789,866.979309,...,844.642395,1031.200562,762.561768,686.991516,703.557739,641.180176,788.218628,734.820862,777.247070,769.877808


In [3]:
##Exporting extraction

out_dir = os.path.expanduser('~/Downloads')
out_csv = os.path.join(out_dir, 'restoration_window_trends_4.csv')

#points_gdf.to_csv(out_csv)

In [2]:
# 1. Setup

import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
import os
import glob
import pandas as pd

# polygons = gpd.read_file('F:/Hotspots/Project shapefiles/hotspots_theilsen_polyg_500k_m2.shp')
polygons = gpd.read_file('F:/Hotspots/Project shapefiles/hotspots_theilsen_savan_100_m2.shp')

raster_files = sorted(glob.glob('F:/Biomass tiffs/Biomass modeling/Georgia/Georgia model only/LOGO_final/*.tif'))

# Initialize an empty list to store results for each polygon
results = []

# 2. Iterate through each raster
for raster_path in raster_files:
    year = os.path.basename(raster_path).split("_")[-1].replace(".tif", "")
    print(f"Processing {year}...")
    
    with rasterio.open(raster_path) as src:
        # Ensure CRS matches
        if polygons.crs != src.crs:
            polygons = polygons.to_crs(src.crs)
            
        # Loop through each polygon
        for idx, row in polygons.iterrows():
            # Use rasterio.mask to clip the raster to the polygon geometry
            # crop=True keeps the resulting array small (the bounding box of the polygon)
            out_image, out_transform = mask(src, [row.geometry], crop=True, nodata=np.nan)
            
            # The mask returns an array [band, height, width]
            data = out_image[0]
            
            # Remove NaNs (NoData pixels)
            valid_pixels = data[~np.isnan(data)]
            
            # Calculate stats
            if valid_pixels.size > 0:
                mean_val = np.nanmean(valid_pixels)
                std_val = np.nanstd(valid_pixels)
            else:
                mean_val = np.nan
                std_val = np.nan
                
            results.append({
                'polygon_id': idx,
                'year': year,
                'mean': mean_val,
                'std': std_val
            })

# 3. Convert results to a DataFrame
stats_df = pd.DataFrame(results)

# Optional: Pivot to wide format if you want one column per year
pivot_df = stats_df.pivot(index='polygon_id', columns='year', values='mean')
print("Extraction complete!")

Processing 1984...
Processing 1985...
Processing 1986...
Processing 1987...
Processing 1988...
Processing 1989...
Processing 1990...
Processing 1991...
Processing 1992...
Processing 1993...
Processing 1994...
Processing 1995...
Processing 1996...
Processing 1997...
Processing 1998...
Processing 1999...
Processing 2000...
Processing 2001...
Processing 2002...
Processing 2003...
Processing 2004...
Processing 2005...
Processing 2006...
Processing 2007...
Processing 2008...
Processing 2009...
Processing 2010...
Processing 2011...
Processing 2012...
Processing 2013...
Processing 2014...
Processing 2015...
Processing 2016...
Processing 2017...
Processing 2018...
Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Extraction complete!


In [4]:
stats_df

,polygon_id,year,mean,std
0,0,1984,738.882141,294.828339
1,1,1984,744.197693,340.830444
2,2,1984,606.434570,212.734116
3,3,1984,433.895142,90.618790
4,4,1984,390.747223,90.497070
...,...,...,...,...
315,3,2023,386.782410,243.060822
316,4,2023,334.495667,154.778107
317,5,2023,250.339340,85.875717
318,6,2023,371.927216,113.549637


In [5]:
##Exporting extraction

out_dir = os.path.expanduser('~/Downloads')
out_csv = os.path.join(out_dir, 'savan_hotspots_trends.csv')

#stats_df.to_csv(out_csv)

In [ ]:
#Fin